# ASG Airlines — End-to-End Data Engineering Pipeline

This notebook ingests the supplied Excel workbook, profiles data quality, cleans and standardizes flight operations data, handles overnight flights, protects PII, creates analytical dimensions/facts, and produces KPI tables for Power BI.

## Business rules
- `6F` flight prefixes are treated as corrupted `6E` (IndiGo) identifiers because the dataset consistently maps `6F` records to IndiGo.
- If arrival timestamp is earlier than departure timestamp, arrival is shifted to the next day.
- Source duration is not trusted for analytics; duration is recalculated from normalized timestamps.
- Exact duplicate flight records are quarantined; the first occurrence is retained for analytics.
- Passenger PII is excluded from Power BI. A restricted masked file is produced for governance demonstration.
- Unusable records are quarantined rather than silently deleted.

In [ ]:
import pandas as pd\npath = '../data/raw/UseCase - Airlines.xlsx'\ntables = pd.read_excel(path, sheet_name=None)\n[(k, v.shape) for k, v in tables.items()]

## 1. Data profiling

In [ ]:
for name, df in tables.items():\n    print(name, df.shape)\n    display(df.head())\n    display(df.isna().sum().to_frame('null_count'))

## 2. Flight transformation and overnight handling

In [ ]:
from src.pipeline import clean_flights\nflights_clean = clean_flights(tables['flights'])\nflights_clean[['flight_id_original','flight_id','airline','departure_datetime','arrival_datetime','is_overnight','flight_duration_minutes']].head(20)

## 3. Quarantine invalid/duplicate records

In [ ]:
quarantine = flights_clean[(flights_clean['is_duplicate']==1) | (~flights_clean['record_valid'])]\nvalid = flights_clean[(flights_clean['is_duplicate']==0) & (flights_clean['record_valid'])]\nprint('Analytical rows:', len(valid))\nprint('Quarantine rows:', len(quarantine))

## 4. KPI generation

In [ ]:
valid.groupby('airline').agg(\n    total_flights=('flight_id','count'),\n    avg_duration_minutes=('flight_duration_minutes','mean'),\n    overnight_flights=('is_overnight','sum'),\n    anomaly_count=('duration_anomaly','sum')\n).reset_index()

## 5. Outputs

In [ ]:
valid.to_csv('../data/processed/fact_flight.csv', index=False)\nquarantine.to_csv('../data/quarantine/flight_quarantine.csv', index=False)